In [ ]:
import os
import io
import json
import shap
import numpy as np
import warnings
import pandas as pd
import boto3
import joblib
import sklearn
from pandas.api.types import is_numeric_dtype
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")

# --- Configuration ---
AWS_S3_BUCKET = "credit-card-fd"
AWS_S3_PREFIX = "cc-fraud-detection-model1"
THRESHOLD = 0.42
TRAIN_XLSX = "Bank_train_retrain_FPFN_feat.xlsx"
TEST_XLSX = "Bank_test_retrain_FPFN_feat.xlsx"
RESULTS_CSV_LOCAL = "fraud_predictions1.csv"

# --- Utilities ---
def read_excel(path: str, engine="openpyxl") -> pd.DataFrame:
    return pd.read_excel(path, engine=engine)

def ensure_columns(df: pd.DataFrame, required: list):
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

def parse_timestamp_features(df: pd.DataFrame, ts_col: str = "timestamp") -> pd.DataFrame:
    """Extracts hour from timestamp if not already a valid numeric time_of_day."""
    if "time_of_day" in df.columns and is_numeric_dtype(df["time_of_day"]):
        df["time_of_day"] = df["time_of_day"].fillna(0).astype(int)
    elif ts_col in df.columns:
        df[ts_col] = pd.to_datetime(df[ts_col], errors="coerce")
        df["time_of_day"] = df[ts_col].dt.hour.fillna(0).astype(int)
    return df

def coerce_binary(df: pd.DataFrame, col: str):
    """
    Convert risk levels or booleans to numeric values.
    """
    if col in df.columns:
        if is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(0).astype(int)
        else:
            # Map LOW, MEDIUM, HIGH and standard booleans
            mapping = {
                "low": 0, "medium": 1, "high": 2,
                "true": 1, "false": 0, "yes": 1, "no": 0, "1": 1, "0": 0
            }
            df[col] = df[col].astype(str).str.lower().map(mapping).fillna(0).astype(int)

def make_onehot_encoder():
    ver = tuple(int(x) for x in sklearn.__version__.split(".")[:2])
    if ver >= (1, 2):
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    else:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def build_preprocessor(num_cols, cat_cols):
    return ColumnTransformer(
        transformers=[
            ("cat", make_onehot_encoder(), cat_cols),
            ("num", "passthrough", num_cols),
        ],
        remainder="drop"
    )

def compute_scale_pos_weight(y_train: pd.Series) -> float:
    pos = (y_train == 1).sum()
    neg = (y_train == 0).sum()
    return float(neg) / float(pos) if pos > 0 else 1.0

def label_error_type(y_true, y_pred):
    out = []
    for t, p in zip(y_true, y_pred):
        if t == 1 and p == 1: out.append("True Positive")
        elif t == 0 and p == 0: out.append("True Negative")
        elif t == 0 and p == 1: out.append("False Positive")
        elif t == 1 and p == 0: out.append("False Negative")
        else: out.append("Unknown")
    return out

def upload_to_s3(local_path: str, bucket: str, key: str):
    s3 = boto3.client("s3")
    s3.upload_file(local_path, bucket, key)
    return f"s3://{bucket}/{key}"

# --- Setup Model Type ---
try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except Exception:
    from sklearn.ensemble import GradientBoostingClassifier
    XGB_AVAILABLE = False

# --- Data Loading ---
train_df = read_excel(TRAIN_XLSX)
test_df = read_excel(TEST_XLSX)

required_train = ["user_id", "card_id", "transaction_id","amount", "merchant_category", "channel", 
                  "country_risk", "time_of_day", "customer_tenure_months", "velocity_1h", "is_fraud", 
                  "country_name", "timestamp","amount_deviation","txn_velocity_1h","geo_distance_km"]
ensure_columns(train_df, required_train)

# --- Preprocessing ---
for df in (train_df, test_df):
    parse_timestamp_features(df, ts_col="timestamp")
    coerce_binary(df, "country_risk")

# --- Column Setup ---
# country_risk and time_of_day are moved to num_cols to preserve their names for SHAP
feature_cols = ["amount", "merchant_category", "channel", "country_risk", "time_of_day", "country_name", "amount_deviation","txn_velocity_1h","geo_distance_km"]
cat_cols = ["merchant_category", "channel", "country_name","country_risk"]
num_cols = ["amount","time_of_day", "amount_deviation","txn_velocity_1h","geo_distance_km"]

X = train_df[feature_cols].copy()
y = train_df["is_fraud"].astype(int).copy()

# Critical Variance Check
print("Feature Unique Values Check:")
print(X.nunique())

scale_pos_weight = compute_scale_pos_weight(y)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# --- Model Building ---
preprocessor = build_preprocessor(num_cols=num_cols, cat_cols=cat_cols)
if XGB_AVAILABLE:
    clf = XGBClassifierXGBClassifier(
        n_estimators=500,           # Increased from 300
        max_depth=9,                # Increased from 7 for deeper feature interaction
        learning_rate=0.05,         # Lowered from 0.08 for more stable convergence
        subsample=0.8,              # Added: uses 80% of data per tree to prevent overfitting
        colsample_bytree=0.8,       # Added: uses 80% of features per tree
        scale_pos_weight=scale_pos_weight, 
        eval_metric="logloss", 
        random_state=42,
        gamma=0.1                   # Added: Minimum loss reduction to make a split
    )
else:
    clf = GradientBoostingClassifier(random_state=42)

model = Pipeline(steps=[("prep", preprocessor), ("clf", clf)])
model.fit(X_train, y_train)

# --- Aggregated SHAP Analysis ---
print("[INFO] Calculating SHAP feature contributions...")
fitted_clf = model.named_steps['clf']
fitted_prep = model.named_steps['prep']
X_val_transformed = fitted_prep.transform(X_val)
feature_names_ohe = fitted_prep.get_feature_names_out()

explainer = shap.TreeExplainer(fitted_clf)
shap_values = explainer.shap_values(X_val_transformed)
mean_abs_shap = np.abs(shap_values[1] if isinstance(shap_values, list) else shap_values).mean(axis=0)

agg_results = {feat: 0.0 for feat in feature_cols}
for i, encoded_name in enumerate(feature_names_ohe):
    for feat in feature_cols:
        if encoded_name.startswith(f"cat__{feat}") or \
           encoded_name.startswith(f"num__{feat}") or \
           encoded_name == feat:
            agg_results[feat] += mean_abs_shap[i]
            break

total_imp = sum(agg_results.values()) or 1.0
feature_importance_df = pd.DataFrame([
    {"feature": k, "contribution_percentage": (v / total_imp) * 100}
    for k, v in agg_results.items()
]).sort_values(by="contribution_percentage", ascending=False)

feature_importance_df.to_csv("feature_contributions_shap_aggregated.csv", index=False)
print(feature_importance_df)

# --- Evaluate on validation (always computed) ---
val_probs = model.predict_proba(X_val)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X_val)
val_pred = (val_probs >= THRESHOLD).astype(int)

val_acc = accuracy_score(y_val, val_pred)

precision = precision_score(y_val, val_pred, average='weighted')
recall = recall_score(y_val, val_pred, average='weighted')
f1 = f1_score(y_val, val_pred, average='weighted')
val_prec, val_rec, val_f1, _ = precision_recall_fscore_support(y_val.to_numpy(), val_pred, average="binary", zero_division=0)
print(f"[VAL] Accuracy={val_acc:.4f}  Precision={precision:.4f}  Recall={recall:.4f}  F1={f1:.4f}")

# --- Score the test set ---
X_test = test_df[feature_cols].copy()
test_probs = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X_test)
test_pred = (test_probs >= THRESHOLD).astype(int)


###below code ended

# --- Metrics on test, only if labels exist ---
metrics = {}
if "is_fraud" in test_df.columns:
    y_test = test_df["is_fraud"].astype(int)
    test_acc = accuracy_score(y_test, test_pred)
    test_prec, test_rec, test_f1, _ = precision_recall_fscore_support(y_test, test_pred, average="binary", zero_division=0)
    cm = confusion_matrix(y_test, test_pred)
    metrics = {
        "accuracy": float(test_acc),
        "precision": float(test_prec),
        "recall": float(test_rec),
        "f1": float(test_f1),
        "confusion_matrix": cm.tolist(),
        "classification_report": classification_report(y_test, test_pred, digits=4, zero_division=0)
    }
    error_type = label_error_type(y_test.tolist(), test_pred.tolist())
else:
    # No ground-truth in test; we can still provide predictions but no test metrics.
    error_type = ["N/A (no ground truth)"] * len(test_pred)

# --- Final Outputs ---
output_df = pd.DataFrame({
    "transaction_id": test_df["transaction_id"].astype(str),
    "user_id": test_df["user_id"].astype(str),
    "amount": test_df["amount"].astype(float),
    "merchant_category": test_df["merchant_category"].astype(str),
    "country_risk": test_df["country_risk"].astype(str),
    "country_name": test_df["country_name"].astype(str),
    "timestamp": test_df["timestamp"],
    "is_fraud_actual": test_df["is_fraud"].astype(int) if "is_fraud" in test_df.columns else 0,
    "fraud_score": test_probs.astype(float),
    "fraud_predicted": test_pred.astype(int)
})
output_df["error_type"] = label_error_type(output_df["is_fraud_actual"], output_df["fraud_predicted"])
output_df.to_csv(RESULTS_CSV_LOCAL, index=False)

# False Positives / Negatives
false_pos_df = output_df[(output_df["is_fraud_actual"] == 0) & (output_df["fraud_predicted"] == 1)]
false_neg_df = output_df[(output_df["is_fraud_actual"] == 1) & (output_df["fraud_predicted"] == 0)]
false_pos_df.to_csv("bank_fraud_false_positives_fd.csv", index=False)
false_neg_df.to_csv("bank_fraud_false_negatives_fd.csv", index=False)

print(f"[INFO] Saved False Positives CSV: {len(false_pos_df)} rows")
print(f"[INFO] Saved False Negatives CSV: {len(false_neg_df)} rows")

# --- S3 Uploads ---
upload_to_s3(RESULTS_CSV_LOCAL, AWS_S3_BUCKET, f"{AWS_S3_PREFIX}/fraud_predictions.csv")
upload_to_s3("bank_fraud_false_positives_fd.csv", AWS_S3_BUCKET, f"{AWS_S3_PREFIX}/bank_fraud_false_positives_FP.csv")
upload_to_s3("bank_fraud_false_negatives_fd.csv", AWS_S3_BUCKET, f"{AWS_S3_PREFIX}/bank_fraud_false_negatives_FN.csv")

print("[INFO] Script execution complete. SHAP contributions and error reports uploaded.")